In [1]:
# from datasets import load_dataset, concatenate_datasets

# df = load_dataset("Qwen/ProcessBench")
# df = concatenate_datasets(df.values()).to_pandas()

# df["split"] = df["id"].str.split("-").str[0]
# df["steps_len"] = df["steps"].str.len()
# df["per_step_len"] = df["steps"].apply(lambda x: [len(y) for y in x])

import pandas as pd

# df = pd.read_parquet("data/processbench1000_length_llm.parquet")
df = pd.read_parquet("data/processbench_with_roi2.parquet")
df.keys()

Index(['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label',
       'split', 'steps_len', 'per_step_len', 'Qwen2.5-Math-PRM-7B',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B', 'roi_step_num', 'roi_step',
       'verbose_roi_step', 'consise_roi_step', 'spelled_eq_roi_step',
       'verbose_steps', 'consise_steps', 'spelled_eq_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--spelled_eq_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--consise_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--verbose_steps',
       'Qwen2.5-Math-PRM-7B--verbose_steps',
       'Qwen2.5-Math-PRM-7B--consise_steps',
       'Qwen2.5-Math-PRM-7B--spelled_eq_steps', 'change_numbers_roi_step',
       'change_numbers_steps',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B--change_numbers_steps',
       'Qwen2.5-Math-PRM-7B--change_numbers_steps', 'verbose_roi_step_v2',
       'verbose_steps_v2', 'Skywork-o1-Open-PRM-Qwen-2.5-7B--verbose_steps_v2',
       'Qwen2.5-Math-PRM-7B--verbose_steps_v2'],
      dtype='o

In [2]:
## Sanity check
row = df.sample().iloc[0]
print(row["label"], row["roi_step_num"])
print("### Steps ###")
print(row["steps"])

print("### Verbose ###")
print(row["verbose_steps_v2"])

print("### Consise ###")
print(row["consise_steps"])

print("### Spelled EQ ###")
print(row["spelled_eq_steps"])

1 1
### Steps ###
["To determine how many bananas the zookeeper needs to order for the next 2 months, we'll break down the requirements for each type of ape (monkeys, gorillas, and baboons) and then sum them up."
 "First, let's calculate the number of bananas needed for the monkeys:\n- Each monkey needs 200 bananas per month.\n- There are 3 monkeys (one for each type of ape).\n- Therefore, the total number of bananas needed for the monkeys is \\( 200 \\text{ bananas/month} \\times 3 \\text{ monkeys} = 600 \\text{ bananas} \\)."
 "Next, let's calculate the number of bananas needed for the gorillas:\n- Each gorilla needs 400 bananas per month.\n- There are 2 gorillas.\n- Therefore, the total number of bananas needed for the gorillas is \\( 400 \\text{ bananas/month} \\times 2 \\text{ gorillas} = 800 \\text{ bananas} \\)."
 "Finally, let's calculate the number of bananas needed for the baboons:\n- Each baboon needs 100 bananas per month.\n- There are 5 baboons.\n- Therefore, the total num

In [ ]:
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer
from utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

```bash
vllm serve hf_cache/Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
vllm serve hf_cache/Qwen--Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8081 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [4]:
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8081/v1"
client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
models = client.models.list()
model = models.data[0].id

In [5]:
print(model)
tokenizer = AutoTokenizer.from_pretrained(model)

hf_cache/Qwen--Qwen2.5-Math-PRM-7B


---

In [43]:
row = df.iloc[75] # QWEN 1944

In [44]:
row

id                                                                                                gsm8k-75
generator                                                                              Qwen2-1.5B-Instruct
problem                                                  Each person in a certain household consumes 0....
steps                                                    [To determine how many weeks a 42 kg bag of ri...
final_answer_correct                                                                                 False
label                                                                                                    1
split                                                                                                gsm8k
steps_len                                                                                                5
per_step_len                                                                     [241, 222, 230, 213, 108]
Qwen2.5-Math-PRM-7B                  

In [45]:
row.problem

'Each person in a certain household consumes 0.2 kg of rice every meal. Supposing 5 members of the household eat rice every lunch and dinner, how many weeks will a 42 kg bag of rice last?'

In [46]:
row.roi_step_num

1

In [47]:
row.label

1

In [40]:
row["consise_steps"]

array(["To solve this problem, we'll break it down into steps. First, let's denote Duncan's age eight years ago as D - 8, and Adam's age four years ago as A - 4.",
       "According to the given information, Duncan's age eight years ago (D - 8) was two times Adam's age four years ago (2 * (A - 4)). We know Duncan's current age is 60. So, let's express Duncan's age eight years ago using his current age: D - 8 = 60 - 8. This simplifies to D - 8 = 52.",
       "Now that we have Duncan's age eight years ago (52), we can set up an equation based on the given information: D - 8 = 2 * (A - 4). Substitute D - 8 with 52: 52 = 2 * (A - 4).",
       "To find Adam's age four years ago (A - 4), divide both sides of the equation by 2: (A - 4) = 52 / 2. This simplifies to (A - 4) = 26.",
       "Solve for A (Adam's age four years ago): A = 26 + 4. This gives us A = 30.",
       "Adam's current age = 34.",
       "Finally, we need to find Adam's age in 8 years: Adam's age in 8 years = Adam's current a

In [41]:
row.steps

array(["To solve this problem, we'll break it down into steps. First, let's denote Duncan's age eight years ago as D - 8, and Adam's age four years ago as A - 4.",
       "According to the given information, Duncan's age eight years ago (D - 8) was two times Adam's age four years ago (2 * (A - 4)). We know Duncan's current age is 60. So, let's express Duncan's age eight years ago using his current age: D - 8 = 60 - 8. This simplifies to D - 8 = 52.",
       "Now that we have Duncan's age eight years ago (52), we can set up an equation based on the given information: D - 8 = 2 * (A - 4). Substitute D - 8 with 52: 52 = 2 * (A - 4).",
       "To find Adam's age four years ago (A - 4), divide both sides of the equation by 2: (A - 4) = 52 / 2. This simplifies to (A - 4) = 26.",
       "Solve for A (Adam's age four years ago): A = 26 + 4. This gives us A = 30.",
       "Since we know Adam's age four years ago, we can find Adam's current age by adding 4: Adam's current age = A + 4. Adam's cur

In [26]:
my_steps = ["To solve this problem, we need to find the number of integers \\(n\\) between 1 and 2002 inclusive for which the function \\(f(n) = f(n + 1)\\), where \\(f(n)\\) is the number of 1's in the binary representation of \\(n\\).",
       "The function \\(f(n)\\) counts the number of 1's in the binary (base-2) representation of an integer \\(n\\). For example:\n- \\(f(5) = f(101_2) = 2\\)\n- \\(f(6) = f(110_2) = 2\\)\n- \\(f(7) = f(111_2) = 3\\)",
       "为了使 f(n) = f(n + 1) 成立，将 n 更改为 n + 1 不应改变二进制表示中 1 的数量。这种情况可能在两种情况下发生：",
       "首先，加 1 会将一个由 0 和 1 组成的序列变成由 1 组成的序列。例如，考虑 \\(1000_2\\)（十进制为 8）。加 1 会得到 \\(1001_2\\)（十进制为 9），这不会改变 1 的数量。",
       "Second, adding 1 changes a sequence of ones into a sequence of zeros plus another one at a higher position. For instance, consider \\(111_2\\) (7 in decimal). Adding 1 results in \\(1000_2\\) (8 in decimal), which also does not change the number of 1's.",
       'We need to find numbers \\(n\\) that satisfy either Condition A or Condition B when considering \\(n\\) and \\(n + 1\\).',
       "First, let's look at sequences of zeros followed by a one. The positions of these sequences are determined by the binary gaps in the binary representation of \\(n\\). A binary gap is defined as the longest sequence of zeros that is surrounded by ones on both sides. For example, \\(1001_2\\) has a binary gap of length 2. For every binary gap in \\(n\\), adding 1 will fill that gap with ones, keeping the number of 1's constant. Thus, every number with a binary gap can potentially satisfy the condition.",
       'This condition occurs right after a sequence of ones in the binary representation. For instance, \\(111_2\\) (7 in decimal) turns into \\(1000_2\\) (8 in decimal). This condition happens less frequently than Condition A since it requires a specific pattern to occur.',
       "To count the numbers, we need to find all \\(n\\) that meet either Condition A or B. We know that \\(f(n) = f(n + 1)\\) happens whenever there is a binary gap in \\(n\\). The longest possible binary gap in a number up to 2002 is 10 (since \\(2^{10} = 1024\\), and we're limited to numbers less than or equal to 2002). We can manually count the numbers satisfying Condition A by considering the binary gap lengths from 1 to 10 and calculating how many numbers have each type of gap.",
       'For Condition B, this happens less frequently and requires a number to end in a sequence of ones. Since 2002 in binary is \\(11111010110_2\\), we can count occurrences manually for sequences of ones at the end of the binary representation.',
       "After completing the counting process (which involves detailed enumeration for each gap size and for sequences of ones), let's say we found that the total number of \\(n\\) satisfying the given condition is \\(X\\).",
       'Given the complexity of manual enumeration for each gap size and sequences of ones, the exact value of \\(X\\) (the number of \\(n\\) where \\(f(n) = f(n + 1)\\)) would require a systematic approach that might involve writing a computer program to efficiently calculate the answer based on the principles outlined. Since I cannot manually enumerate all possibilities, my final response for \\(X\\) must be given as "X", acknowledging that a precise numerical answer would be the result of executing a detailed algorithmic process based on the reasoning provided.',
       'Thus, the final answer, represented symbolically due to the complexity of direct calculation, is \\(\\boxed{X}\\), where \\(X\\) represents the total count of such numbers \\(n\\) between 1 and 2002.']

In [27]:
input_ids, token_mask = prepare_input(
                                model, 
                                problem=row.problem, 
                                steps=my_steps, 
                                tokenizer=tokenizer,
                                convert_to_list=True
                        )

In [28]:
print(tokenizer.decode(input_ids))

<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
For any positive integer $n$, let $f(n)$ denote the number of 1's in the base-2 representation of $n$. For how many values of $n$ with $1 \leq n \leq 2002$ do we have $f(n)=f(n+1)$?<|im_end|>
<|im_start|>assistant
To solve this problem, we need to find the number of integers \(n\) between 1 and 2002 inclusive for which the function \(f(n) = f(n + 1)\), where \(f(n)\) is the number of 1's in the binary representation of \(n\).<extra_0>The function \(f(n)\) counts the number of 1's in the binary (base-2) representation of an integer \(n\). For example:
- \(f(5) = f(101_2) = 2\)
- \(f(6) = f(110_2) = 2\)
- \(f(7) = f(111_2) = 3\)<extra_0>为了使 f(n) = f(n + 1) 成立，将 n 更改为 n + 1 不应改变二进制表示中 1 的数量。这种情况可能在两种情况下发生：<extra_0>首先，加 1 会将一个由 0 和 1 组成的序列变成由 1 组成的序列。例如，考虑 \(1000_2\)（十进制为 8）。加 1 会得到 \(1001_2\)（十进制为 9），这不会改变 1 的数量。<extra_0>Second, adding 1 changes a sequence of ones into a se

In [29]:
logits = client.embeddings.create(
        input=input_ids,
        model=model,
    )

In [30]:
import torch
derive_step_rewards_vllm(
    model,
    logits,
    torch.Tensor([token_mask]),
    tokenizer
)#[0][3]

[[0.99609375,
  0.98828125,
  0.875,
  0.68359375,
  0.9453125,
  0.9296875,
  0.380859375,
  0.85546875,
  0.40234375,
  0.7578125,
  0.42578125,
  0.271484375,
  0.6171875]]

In [ ]:
[[0.99609375,
  0.98828125,
  0.85546875,
  0.58984375,
  0.8828125,
  0.93359375,
  0.322265625,
  0.83203125,
  0.376953125,
  0.734375,
  0.3984375,
  0.255859375,
  0.61328125]]